# RF Model train
This notebook replicates the explainable model approach from the paper by Stephan Matzka (2020).

In [95]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.12/bin/python3


In [96]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [97]:
from Scripts.RandomForestTrain import trainRF
from Scripts.DecisionTreeTrain import train_explainable_tree, evaluate_tree, plot_decision_tree


In [98]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from Scripts.RUSBoostTrain import train_rusboost

%load_ext autoreload
%autoreload 2

CSV_PATH = "Data_file/ai4i2020.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [99]:
#model, X_test, y_test, y_pred_rf, y_proba_rf, results = trainRF("configs/RandomForest/RFtrain.yaml")
rf_raw, rf_calibrated, X_test, y_test, y_pred_rf, y_proba_rf, results = trainRF("configs/RandomForest/RFtrain.yaml")
# Train Explainable Decision Tree using RF


Chosen Threshold: 0.01


AttributeError: 'float' object has no attribute 'astype'

In [100]:
print("Chosen threshold:", results["threshold"])
print("PR-AUC:", results["pr_auc"])
print("ROC-AUC:", results["roc_auc"])

Chosen threshold: 0.01
PR-AUC: 0.715609070185756
ROC-AUC: 0.9576939471440751


In [101]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, digits=4))

print("\nROC AUC:", roc_auc_score(y_test, y_proba_rf))
print("Average Precision (PR AUC):", average_precision_score(y_test, y_proba_rf))

print("\nChosen Threshold:", results["threshold"])

Confusion Matrix:
[[1529  403]
 [   3   65]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9980    0.7914    0.8828      1932
           1     0.1389    0.9559    0.2425        68

    accuracy                         0.7970      2000
   macro avg     0.5685    0.8736    0.5627      2000
weighted avg     0.9688    0.7970    0.8610      2000


ROC AUC: 0.9576939471440751
Average Precision (PR AUC): 0.715609070185756

Chosen Threshold: 0.01


In [102]:
# =========================================================
# Create timestamped experiment folder + save artifacts
# =========================================================

import os
import yaml
import joblib
from datetime import datetime
import json

# --- 1. Create timestamp ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# --- 2. Define base experiment directory ---
base_dir = "experiments_rf"
run_dir = os.path.join(base_dir, f"run_{timestamp}")

# --- 3. Create directory ---
os.makedirs(run_dir, exist_ok=True)

print(f"Experiment directory created: {run_dir}")

# --- 4. Save trained model ---
model_path = os.path.join(run_dir, "random_forest.pkl")
#joblib.dump(model, model_path)
joblib.dump(rf_raw, os.path.join(run_dir, "rf_raw.pkl"))
joblib.dump(rf_calibrated, os.path.join(run_dir, "rf_calibrated.pkl"))

print(f"Model saved to: {model_path}")

# --- 5. Save config file ---
config_src = "configs/RandomForest/RFtrain.yaml"
config_dst = os.path.join(run_dir, "RFtrain.yaml")

with open(config_src, "r") as f:
    config_data = yaml.safe_load(f)

with open(config_dst, "w") as f:
    yaml.dump(config_data, f)

print(f"Config saved to: {config_dst}")

# --- 6. (Optional) Save metrics ---
from sklearn.metrics import confusion_matrix, classification_report

metrics = {
    "model": "RandomForest",
    "confusion_matrix": confusion_matrix(y_test, y_pred_rf).tolist(),
    "classification_report": classification_report(y_test, y_pred_rf, output_dict=True),
    "roc_auc": roc_auc_score(y_test, y_proba_rf),
    "pr_auc": average_precision_score(y_test, y_proba_rf),
    "threshold": results["threshold"]
}

metrics_path = os.path.join(run_dir, "metrics.json")

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Metrics saved to: {metrics_path}")


notes_path = os.path.join(run_dir, "notes.txt")

with open(notes_path, "w") as f:
    f.write(
        "Experiment Notes\n"
        "=================\n\n"
        f"Timestamp: {timestamp}\n\n"
        "Observations:\n- \n\n"
        "Changes Made:\n- \n\n"
        "Next Steps:\n- \n"
    )
    print(f"Notes file created at: {notes_path}")

Experiment directory created: experiments_rf/run_2026-04-21_18-02-06
Model saved to: experiments_rf/run_2026-04-21_18-02-06/random_forest.pkl
Config saved to: experiments_rf/run_2026-04-21_18-02-06/RFtrain.yaml
Metrics saved to: experiments_rf/run_2026-04-21_18-02-06/metrics.json
Notes file created at: experiments_rf/run_2026-04-21_18-02-06/notes.txt


In [8]:

selected_uids = {
    "TWF": [2672, 3866, 6341, 8358, 9019],
    "HDF": [3237, 4079, 4174, 4327, 4502],
    "PWF": [464, 1493, 3001, 7537, 8583],
    "OSF": [250, 3020, 5400, 7592, 9660],
}

all_selected_uids = [uid for group in selected_uids.values() for uid in group]

selected_cases = df[df["UDI"].isin(all_selected_uids)].copy().sort_values("UDI")

selected_cases[["UDI", "Type", "Machine failure", "TWF", "HDF", "PWF", "OSF"]]

,UDI,Type,Machine failure,TWF,HDF,PWF,OSF
249,250,L,1,0,0,0,1
463,464,L,1,0,0,1,0
1492,1493,M,1,0,0,1,0
2671,2672,M,1,1,0,0,0
3000,3001,H,1,0,0,1,0
3019,3020,L,1,0,0,0,1
3236,3237,M,1,0,1,0,0
3865,3866,H,1,1,0,0,0
4078,4079,H,1,0,1,0,0
4173,4174,M,1,0,1,0,0
